In [ ]:
import os
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


# FRED_API_KEY is read from the environment; do not store credentials here.

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

from src import config
from src.data.options import run_screen, concat_results

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# IV Subsumption — Liquid Fixed-Income ETF Universe

**Research question**: Does 30-day options-implied volatility (`iv_30d`) subsume
the realized-volatility fragility signal (`vol_12w`), or does `vol_12w` carry
incremental predictive power over IV for forward drawdowns?

**Universe**: Tickers from the core panel that pass the option liquidity screen
(dollar vega, theta/vega, dollar gamma filters). ~38 tickers.

**Data**: IV computed from ThetaData EOD option prices via Black-Scholes inversion.
Date range: 2020-01-03 to present (quarterly snap dates for screen; weekly Fridays for IV).

**Outcome**: `fwd_maxdd_12w` — maximum drawdown over the 12 weeks following each observation.

In [ ]:
# ── Option liquidity screen ─────────────────────────────────────────────────
# Fetches full call chain for all core-panel tickers on quarterly snap dates,
# computes BSM Greeks, applies liquidity filters. Results are JSON-cached.

SNAP_DATES = [
    "2020-01-02", "2020-04-01", "2020-07-01", "2020-10-01",
    "2021-01-04", "2021-04-01", "2021-07-01", "2021-10-01",
    "2022-01-03", "2022-04-01", "2022-07-01", "2022-10-03",
    "2023-01-03", "2023-04-03", "2023-07-03", "2023-10-02",
    "2024-01-02", "2024-04-01", "2024-07-01", "2024-10-01",
    "2025-01-02", "2025-04-01",
]

from src.data.options_universe import UNIVERSE

#all_tickers = pd.read_csv(config.CORE_PANEL_CSV)["Symbol"].unique().tolist()
screen_results = run_screen(list(UNIVERSE.keys()), snap_dates=SNAP_DATES, rights=("C", "P"))
concat_results()

ticker_summary = screen_results["ticker_summary"]
liquid_tickers = ticker_summary[ticker_summary["liquid"]]["ticker"].tolist()
print(f"Liquid tickers ({len(liquid_tickers)}): {sorted(liquid_tickers)}")
display(ticker_summary.sort_values("mean_pass_rate", ascending=False))

In [ ]:
# ── Load IV panel ───────────────────────────────────────────────────────────
# Built by: python scripts/04_build_iv_panel.py
PANEL_PATH = config.PROCESSED_DIR / "options_screen" / "iv_panel_full.csv"
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"IV panel not found at {PANEL_PATH}.\n"
        "Run: python scripts/04_build_iv_panel.py"
    )

panel = pd.read_csv(PANEL_PATH, parse_dates=["date"])
print(f"Loaded {len(panel):,} rows | {panel['ticker'].nunique()} tickers")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")
print()
print(panel.groupby('ticker')[['vol_12w_annualized', 'iv_30d', 'iv_realized_spread']]
      .describe().round(4))

In [ ]:
panel.isna().sum()

## 1. Descriptive Statistics

In [ ]:
STAT_COLS = ['vol_12w_annualized', 'iv_30d', 'iv_realized_spread',
             'fwd_maxdd_12w', 'fwd_ret_4w', 'fwd_vol_12w']

disp = (
    panel.dropna(subset=['iv_30d'])
    .groupby('ticker')[STAT_COLS]
    .describe(percentiles=[.1, .25, .5, .75, .9])
    .T
    .round(4)
)
display(disp)

## 2. Exploratory Plots

In [ ]:
# ── IV vs realised vol over time ────────────────────────────────────────────
tickers = sorted(panel['ticker'].unique())

STRESS_START = pd.Timestamp('2022-01-01')
STRESS_END   = pd.Timestamp('2022-12-31')

fig, axes = plt.subplots(len(tickers), 1, figsize=(10, 4 * len(tickers)), sharex=False)
if len(tickers) == 1:
    axes = [axes]

for ax, ticker in zip(axes, tickers):
    sub = panel[panel['ticker'] == ticker].dropna(subset=['iv_30d', 'vol_12w_annualized'])
    ax.plot(sub['date'], sub['iv_30d'],             label='iv_30d (ATM 30d)', color='steelblue', lw=1.5)
    ax.plot(sub['date'], sub['vol_12w_annualized'], label='vol_12w ann.',     color='darkorange', lw=1.5, ls='--')
    x0 = max(sub['date'].min(), STRESS_START)
    x1 = min(sub['date'].max(), STRESS_END)
    if x0 < x1:
        ax.axvspan(x0, x1, alpha=0.12, color='red', label='2022 stress')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_title(f'{ticker}: IV vs. Realised Volatility')
    ax.legend(fontsize=8)

plt.tight_layout()
(ROOT / 'docs/figures').mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / 'docs/figures/iv_vs_realised.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── IV-realized spread over time ────────────────────────────────────────────
fig, axes = plt.subplots(len(tickers), 1, figsize=(10, 3.5 * len(tickers)), sharex=False)
if len(tickers) == 1:
    axes = [axes]

for ax, ticker in zip(axes, tickers):
    sub = panel[panel['ticker'] == ticker].dropna(subset=['iv_realized_spread'])
    colors = sub['iv_realized_spread'].apply(lambda x: 'steelblue' if x >= 0 else 'salmon')
    ax.bar(sub['date'], sub['iv_realized_spread'], color=colors, width=5)
    ax.axhline(0, color='black', lw=0.8)
    x0 = max(sub['date'].min(), STRESS_START)
    x1 = min(sub['date'].max(), STRESS_END)
    if x0 < x1:
        ax.axvspan(x0, x1, alpha=0.12, color='red', label='2022 stress')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_title(f'{ticker}: IV - Realised Spread (iv_30d - vol_12w_ann.)')
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(ROOT / 'docs/figures/iv_realized_spread.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Scatter: iv_30d vs vol_12w_annualized ───────────────────────────────────
fig, axes = plt.subplots(1, len(tickers), figsize=(6 * len(tickers), 5))
if len(tickers) == 1:
    axes = [axes]

for ax, ticker in zip(axes, tickers):
    sub = panel[panel['ticker'] == ticker].dropna(subset=['iv_30d', 'vol_12w_annualized'])
    ax.scatter(sub['vol_12w_annualized'], sub['iv_30d'], alpha=0.5, s=18, color='steelblue')
    lo = min(sub['vol_12w_annualized'].min(), sub['iv_30d'].min()) * 0.9
    hi = max(sub['vol_12w_annualized'].max(), sub['iv_30d'].max()) * 1.1
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, label='IV = vol')
    corr = sub[['vol_12w_annualized', 'iv_30d']].corr().iloc[0, 1]
    ax.set_xlabel('vol_12w (annualised)')
    ax.set_ylabel('iv_30d')
    ax.set_title(f'{ticker}  ρ = {corr:.2f}')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

plt.tight_layout()
fig.savefig(ROOT / 'docs/figures/iv_vs_realised_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Regression Analysis

**Outcome**: `fwd_maxdd_12w` (maximum 12-week forward drawdown, ≤ 0).

Three specifications:
1. `vol_12w_annualized` alone
2. `iv_30d` alone
3. Both jointly (subsumption test)

**Standard errors**: Two-way clustering by ticker and date (CGM 2011).
With ~38 tickers and ~250 weekly dates, both dimensions are well-identified.
All specifications include ticker fixed effects (`C(ticker)`).

In [ ]:
# ── Regression dataset ──────────────────────────────────────────────────────
reg = panel.dropna(subset=['fwd_maxdd_12w', 'vol_12w_annualized', 'iv_30d']).copy()
reg['ticker'] = reg['ticker'].astype(object)
reg['year'] = reg['date'].dt.year

print(f'Regression rows: {len(reg):,}')
print(reg.groupby('ticker').size().rename('obs'))

In [ ]:
# ── CGM two-way cluster helper ──────────────────────────────────────────────
def twoway_cluster_se(formula, data, col1, col2):
    """CGM (2011) two-way cluster SE: V = V(col1) + V(col2) - V(HC1)."""
    model = smf.ols(formula, data=data)
    r_c1  = model.fit(cov_type='cluster', cov_kwds={'groups': data[col1]})
    r_c2  = model.fit(cov_type='cluster', cov_kwds={'groups': data[col2]})
    r_hc1 = model.fit(cov_type='HC1')
    V = (r_c1.cov_params().values
         + r_c2.cov_params().values
         - r_hc1.cov_params().values)
    eigvals, eigvecs = np.linalg.eigh(V)
    V_psd = eigvecs @ np.diag(np.maximum(eigvals, 0)) @ eigvecs.T
    se_2way = np.sqrt(np.diag(V_psd))
    return r_c1, se_2way, V_psd


FE   = 'C(ticker)'
kw_d = lambda df: dict(cov_type='cluster', cov_kwds={'groups': df['date']})

m1 = smf.ols(f'fwd_maxdd_12w ~ vol_12w_annualized + {FE}', data=reg).fit(**kw_d(reg))
m2 = smf.ols(f'fwd_maxdd_12w ~ iv_30d + {FE}', data=reg).fit(**kw_d(reg))
m3 = smf.ols(f'fwd_maxdd_12w ~ vol_12w_annualized + iv_30d + {FE}', data=reg).fit(**kw_d(reg))

tbl = summary_col(
    [m1, m2, m3],
    model_names=['(1) vol only', '(2) iv only', '(3) joint'],
    stars=True,
    float_format='%.4f',
    info_dict={'N': lambda m: f'{int(m.nobs):,}', 'R²': lambda m: f'{m.rsquared:.4f}'},
    regressor_order=['vol_12w_annualized', 'iv_30d'],
)
print(tbl)

In [ ]:
# ── Incremental R² ──────────────────────────────────────────────────────────
r2_1, r2_2, r2_3 = m1.rsquared, m2.rsquared, m3.rsquared

print(f'R²: (1) vol only = {r2_1:.4f}')
print(f'R²: (2) iv only  = {r2_2:.4f}')
print(f'R²: (3) joint    = {r2_3:.4f}')
print()
print(f'ΔR²  (3) vs (1): {r2_3 - r2_1:+.4f}  (IV adds {(r2_3-r2_1)/r2_1*100:.1f}% to vol baseline)')
print(f'ΔR²  (3) vs (2): {r2_3 - r2_2:+.4f}  (vol adds {(r2_3-r2_2)/r2_2*100:.1f}% to IV baseline)')

In [ ]:
# ── Coefficient summary with t-stats ────────────────────────────────────────
joint_vars = [v for v in ['vol_12w_annualized', 'iv_30d'] if v in m3.params.index]

for label, model, key_vars in [
    ('(1) vol only', m1, ['vol_12w_annualized']),
    ('(2) iv only',  m2, ['iv_30d']),
    ('(3) joint',    m3, joint_vars),
]:
    print(f'\n{label}')
    print(pd.DataFrame({
        'coef':    model.params[key_vars],
        't_stat':  model.tvalues[key_vars],
        'p_value': model.pvalues[key_vars],
    }).round(4))

In [ ]:
# ── CGM two-way robustness (ticker × date) ─────────────────────────────────
f3 = f'fwd_maxdd_12w ~ vol_12w_annualized + iv_30d + {FE}'
r_cgm3, se_cgm3, _ = twoway_cluster_se(f3, reg, 'ticker', 'date')

params_cgm = r_cgm3.params[joint_vars]
se_cgm     = pd.Series(se_cgm3, index=r_cgm3.params.index)[joint_vars]
t_cgm      = params_cgm / se_cgm

print('Spec (3) — CGM two-way SE (ticker × date):')
print(pd.DataFrame({'coef': params_cgm, 'se_cgm': se_cgm, 't_cgm': t_cgm}).round(4))

## 4. Interpretation

**Subsumption test (spec 3 vs 1 and 2)**
If `vol_12w_annualized` remains statistically significant in spec 3, realized
volatility carries incremental predictive power over IV — it is not subsumed.
If it becomes insignificant while `iv_30d` stays significant, IV fully captures
the realized signal.

**Interpretation of positive IV coefficient**
A positive `iv_30d` coefficient in the joint spec is consistent with a variance
risk premium: higher IV predicts larger forward drawdowns beyond what realized
vol already explains, reflecting compensation for tail risk embedded in options prices.

See `docs/iv_pilot_notes.md` for full methodology.